# Test Agent to Agent Protocol
First discover the agent capabilites by getting the Agent Card, then submit the tasks that has corresponding capabilities in that agent.

In [1]:
import requests

AGENT_BASE_URL = "http://localhost:8081"


def discover():
    return requests.get(f"{AGENT_BASE_URL}/.well-known/agent.json").json()


def task(prompt):
    return requests.post(
        f"{AGENT_BASE_URL}/task",
        json={
            "id": "task-001",
            "input": prompt,
            "context": {}
        }
    ).json()


if __name__ == "__main__":
    agent = discover()

    # -------------------------
    # PRINT FULL AGENT CARD
    # -------------------------
    print("FULL AGENT CARD:\n")
    print(agent)

    print("\n====================\n")

    # -------------------------
    # PRINT CAPABILITIES
    # -------------------------
    print("Capabilities:")
    for c in agent["capabilities"]:
        print("-", c["domain"])

    # -------------------------
    # TASK REGISTRY (DOMAIN → PROMPT)
    # -------------------------
    task_registry = [
        ("hr", "Get leave balance for Sara Ali"),
        ("weather", "What is the weather expected in Dubai today?"),
        ("orders", "Show last 10 Kafka orders"),
        ("kafka", "Show last 5 Kafka orders"),
        ("hr", "Basic profile for Osama Oransa"),
        ("hr", "Get leave balance for EMP002"),
        ("hr", "Remote work policy"),
    ]

    # -------------------------
    # AUTO EXECUTE MATCHING TASKS (CORRECT)
    # -------------------------
    for key, prompt in task_registry:
        matched = False
    
        for capability in agent["capabilities"]:
            domain = capability["domain"].lower()
            tokens = domain.split()
    
            if key in tokens:
                print(f"\n{key.upper()} capability found → sending task: {prompt}\n")
    
                result = task(prompt)
                print(f"{key.upper()} RESULT:")
                print(result)
    
                matched = True
                break
    
        if not matched:
            print(f"\n[SKIP] No capability found for task: {key}")

FULL AGENT CARD:

{'name': 'Dynamic MCP Orchestrator', 'description': 'Orchestrates MCP tool ecosystem dynamically.', 'capabilities': [{'domain': 'Weather MCP Server', 'description': 'Weather services and historic climate lookups', 'examples': ['Weather expected in Dubai today', 'Weather expected in Dubai on 2026-06-29']}, {'domain': 'HR MCP Server', 'description': 'HR services and employee operations', 'examples': ['Remote Work Policy?', 'Get basic profile for Osama Oransa', 'What is the current leave balance for Sara Ali?']}]}


Capabilities:
- Weather MCP Server
- HR MCP Server

HR capability found → sending task: Get leave balance for Sara Ali

HR RESULT:
{'id': 'task-001', 'status': 'completed', 'output': '{\n  "employee_code": "EMP002",\n  "annual_leave_days": 25,\n  "sick_leave_days": 10,\n  "parental_leave_days": 5\n}', 'artifacts': {'tool_trace': [{'tool': 'get_employee_code', 'server': 'HR MCP Server'}, {'tool': 'get_leave_balance', 'server': 'HR MCP Server'}], 'raw': {'answe